# [Introduction to Data Science](http://datascience-intro.github.io/1MS041-2026/)    
## 1MS041, 2026 
&copy;2026 Raazesh Sainudiin, Benny Avelin. [Attribution 4.0 International     (CC BY 4.0)](https://creativecommons.org/licenses/by/4.0/)

# ProbSS 5: Markov chains (compute, classify, learn from data)

This notebook has three parts, each with one message.

1. **Compute.** A distribution is a row vector and moves *forward* ($\mu_{t+1} = \mu_t P$). A function is a column vector and moves *backward* ($(Pf)(x) = \mathrm{E}_x[f(X_1)]$).
2. **Classify.** Irreducibility and the period depend only on which entries of $P$ are positive. Together they decide what the stationary distribution tells us about the long run.
3. **Learn from data.** Estimating $P$ is just counting. The real test is to simulate the fitted chain and compare it with data it has not seen.

The worksheet exercises are referred to by their numbers on the Problem Session 5 sheet. Solve them by hand first; the cells here are for checking.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from math import gcd

from pathlib import Path

def course_data(filename):
    candidates = (
        Path("data") / filename,
        Path("master/jp/data") / filename,
    )
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"Could not find {filename}. Run this notebook from master/jp "
        "or from the repository root."
    )

np.set_printoptions(precision=4, suppress=True)

## 1. Two ways to compute with $P$ (worksheet exercise 1)

For the weather chain we compute $\mathrm{E}_{\mathrm D}[\#\text{wet days among } X_1, X_2, X_3] = \sum_{t=1}^{3} \mu_0 P^t f$ with $f = \mathbf 1_{\{\mathrm W\}}$.

NumPy's 1-D arrays do not know whether they are rows or columns. **The side of `@` decides:** `mu @ P` treats `mu` as a row, and `P @ f` treats `f` as a column.

In [ ]:
P_weather = np.array([[3/4, 1/4],     # states: 0 = D (dry), 1 = W (wet)
                      [1/2, 1/2]])
mu0 = np.array([1.0, 0.0])            # start dry: a distribution (row)
f = np.array([0.0, 1.0])              # f = 1{W}: a function (column)

# Forward: push the distribution, then evaluate f.
forward, mu = 0.0, mu0
for t in range(3):
    mu = mu @ P_weather
    forward += mu @ f

# Backward: pull the function, then evaluate at the start.
backward, g = np.zeros(2), f
for t in range(3):
    g = P_weather @ g
    backward += g

print("forward  (start D only):", forward, "  exact 57/64 =", 57/64)
print("backward (both starts): ", backward, "  exact [57/64, 39/32] =", [57/64, 39/32])

**Exercise 1(c): time-dependent $P$.** The law of $X_2$ is $\mu_0 P_1 P_2$, and $\mathrm{E}_x[f(X_2)] = (P_1 P_2 f)(x)$. In the backward computation, the *last* transition $P_2$ is applied *first*.

In [ ]:
P1 = np.array([[0.0, 1.0], [1.0, 0.0]])
P2 = np.array([[1.0, 0.0], [0.5, 0.5]])
f1 = np.array([1.0, 0.0])                      # f = 1{state 0}

print("law of X2, P1 then P2:", mu0 @ P1 @ P2)
print("law of X2, P2 then P1:", mu0 @ P2 @ P1, " (order matters)")

g = P2 @ f1        # first pull back through the LAST step
g = P1 @ g         # then through the first step
print("E_x f(X2), backward:  ", g, "  forward check from x=0:", mu0 @ P1 @ P2 @ f1)

**Exercises 23–24: two-step probabilities.** `P**2` squares every entry, while `P @ P` is the two-step matrix. A quick sanity check catches the bug: the rows of a stochastic matrix must sum to 1.

In [ ]:
DOCK, SORT, DISPATCH = 0, 1, 2
P_hat = np.array([[0.20, 0.80, 0.00],
                  [0.10, 0.40, 0.50],
                  [0.25, 0.50, 0.25]])

print("row sums of P**2:  ", (P_hat**2).sum(axis=1), " <- not a transition matrix")
print("row sums of P @ P: ", (P_hat @ P_hat).sum(axis=1))
print("Exercise 23: P(X2 = Dispatch | X0 = Dock) =", (P_hat @ P_hat)[DOCK, DISPATCH])

## 2. Irreducibility and periodicity (exercises 25, 25b, 25c)

The long-run theory rests on two conditions:

* **Irreducible:** every state can reach every other state. Then the stationary distribution $\pi$ is unique, and the fraction of time one path spends in $x$ converges to $\pi(x)$.
* **Aperiodic** (in addition): the period $d = \gcd\{k \ge 1 : P^k(x,x) > 0\}$ equals 1. Only then does the law of $X_t$ converge to $\pi$ from every start (exercise 27).

Both conditions only depend on **which** entries of $P$ are positive. Draw an arrow $x \to y$ whenever $P(x,y) > 0$.

* `is_irreducible` collects, for each start $x$, all states reachable from $x$ (`reachable_from`) and checks that this is all $n$ states.
* `period` computes the gcd of the return times of state 0 by checking $P^k(0,0) > 0$ for $k = 1, \dots, 3n$. For an irreducible chain all states have the same period, so one state suffices. Checking only up to $k = n$ is **not** enough (exercise 25b).

In [ ]:
def reachable_from(P, start):
    # All states that can be reached from `start` by following arrows x -> y with P[x, y] > 0.
    n = len(P)
    reached = {start}
    to_visit = [start]
    while to_visit:
        x = to_visit.pop()
        for y in range(n):
            if P[x, y] > 0 and y not in reached:
                reached.add(y)
                to_visit.append(y)
    return reached

def is_irreducible(P):
    # Irreducible: from EVERY state we can reach ALL n states.
    n = len(P)
    for x in range(n):
        if len(reachable_from(P, x)) < n:
            return False
    return True

def period(P, x=0):
    # gcd of all k with P^k(x, x) > 0. Checking k = 1, ..., 3n is enough.
    n = len(P)
    return_times = []
    Pk = np.eye(n)
    for k in range(1, 3 * n + 1):
        Pk = Pk @ P                      # now Pk = P^k
        if Pk[x, x] > 0:
            return_times.append(k)
    return gcd(*return_times)    

mats = {
    "25 A": [[.1,.4,.2,.2,.1],[.2,.1,.3,.3,.1],[.1,.2,.2,.3,.2],[.2,.2,.2,.1,.3],[.3,.1,.2,.2,.2]],
    "25 B": [[0,.5,.5,0,0],[0,0,0,.4,.6],[0,0,0,.7,.3],[1,0,0,0,0],[1,0,0,0,0]],
    "25 C": [[.3,.3,.2,.1,.1],[.2,.3,.3,.1,.1],[.1,.2,.4,.2,.1],[0,0,0,.5,.5],[0,0,0,.4,.6]],
    "25 D": [[0,0,.5,.3,.2],[0,0,.2,.5,.3],[.6,.4,0,0,0],[.5,.5,0,0,0],[.3,.7,0,0,0]],
    "25b P": [[0,1,0,0],[.5,0,.5,0],[0,0,0,1],[0,1,0,0]],
    "25b R": [[1/3,1/3,1/3],[0,.5,.5],[0,.5,.5]],
    "25c": [[0,1,0,0,0,0],[0,0,1,0,0,0],[0,0,0,.5,.5,0],[1,0,0,0,0,0],[0,0,0,0,0,1],[0,0,0,1,0,0]],
}
for name, M in mats.items():
    M = np.array(M)
    if not is_irreducible(M):
        print(f"{name:6}: not irreducible")
    else:
        print(f"{name:6}: irreducible, period = {period(M)}")

### The stationary distribution (exercise 26)

$\pi P = \pi$ is a **left** eigenvector equation, but `np.linalg.solve` wants a column vector of unknowns. Transposing gives $(P - I)^\top \pi = 0$. These $n$ equations are redundant, so we replace the last one by $\pi(0) + \dots + \pi(n-1) = 1$.

In [ ]:
def stationary(P):
    n = len(P)
    M = (P - np.eye(n)).T                # (P - I)^T pi = 0
    b = np.zeros(n)
    M[-1, :] = 1.0                       # replace the last equation ...
    b[-1] = 1.0                          # ... by pi(0) + ... + pi(n-1) = 1
    return np.linalg.solve(M, b)

pi = stationary(P_hat)
print("Exercise 26: pi =", pi, "  pi_Sort =", round(pi[SORT], 4))
print("check pi P = pi:", np.allclose(pi @ P_hat, pi))

## 3. Learning a chain from rainfall data

Daily rainfall in Christchurch is reduced to two states: dry (0 mm) and wet (> 0 mm). The plan has three steps:

1. **Estimate** $P$ from the first 80% of the days (the *training* period) by counting transitions.
2. **Simulate** the fitted chain for as many days as the remaining 20% (the *test* period).
3. **Compare** the simulated weather with the true weather in the test period, and with a model without memory (independent days).

For simplicity we treat the records as consecutive days and ignore the few missing dates.

In [ ]:
rain = pd.read_csv(
    course_data("rainfallInChristchurch.csv"),
    names=["date_code", "rain_mm"],
)
rain["rain_mm"] = pd.to_numeric(rain["rain_mm"], errors="coerce")    # unreadable entries -> NaN
rain = rain.dropna().sort_values("date_code").drop_duplicates("date_code")

wet = (rain["rain_mm"] > 0).astype(int).to_numpy()                  # state 0 = dry, 1 = wet
print("number of days:      ", len(wet))
print("fraction of wet days:", round(wet.mean(), 3))

### Step 1: estimate $P$ on the training period

$\hat P(x,y) = \dfrac{\#\{\text{days with state } x \text{ followed by state } y\}}{\#\{\text{days with state } x \text{ that have a next day}\}}$.

In [ ]:
def estimate_P(states):
    # Count the transitions today -> tomorrow and divide each row by its total.
    counts = np.zeros((2, 2))
    for today, tomorrow in zip(states[:-1], states[1:]):
        counts[today, tomorrow] += 1
    return counts / counts.sum(axis=1, keepdims=True)

split = int(0.8 * len(wet))
train = wet[:split]
test = wet[split:]

P_rain = estimate_P(train)
print("estimated P:\n", P_rain.round(3))
print("stationary distribution:", stationary(P_rain).round(3))

### Step 2: simulate the fitted chain, and a model without memory

The Markov simulation draws tomorrow from the row of today. The model without memory draws every day independently: wet with probability equal to the wet fraction of the training period.

In [ ]:
rng = np.random.default_rng(2026)

def simulate_chain(P, x0, T, rng):
    # One path of length T started in x0: draw the next state from the row of the current state.
    path = [x0]
    for _ in range(T - 1):
        path.append(rng.choice(len(P), p=P[path[-1]]))
    return np.array(path)

markov_sim = simulate_chain(P_rain, x0=test[0], T=len(test), rng=rng)
iid_sim = (rng.random(len(test)) < train.mean()).astype(int)

n_show = 150
fig, ax = plt.subplots(figsize=(10, 2))
ax.imshow([test[:n_show], markov_sim[:n_show], iid_sim[:n_show]], aspect="auto", cmap="Blues_r")
ax.set_yticks([0, 1, 2], ["observed", "Markov", "no memory"])
ax.set_xlabel("day in test period (dark = dry, white = wet)")
plt.show()

### Step 3: compare summary statistics

A **spell** is a maximal run of consecutive wet (or dry) days. The Markov chain can create long spells because today's weather influences tomorrow's; the model without memory cannot.

In [ ]:
def spell_lengths(x, state):
    # Lengths of all maximal runs of `state` in the 0/1 sequence x.
    lengths, run = [], 0
    for value in x:
        if value == state:
            run += 1
        elif run > 0:
            lengths.append(run)
            run = 0
    if run > 0:
        lengths.append(run)
    return np.array(lengths)

def summary(x):
    return {
        "fraction wet": x.mean(),
        "P(wet | wet yesterday)": estimate_P(x)[1, 1],
        "mean dry spell (days)": spell_lengths(x, 0).mean(),
        "mean wet spell (days)": spell_lengths(x, 1).mean(),
        "longest dry spell (days)": spell_lengths(x, 0).max(),
        "longest wet spell (days)": spell_lengths(x, 1).max(),
    }

pd.DataFrame({
    "observed (test)": summary(test),
    "Markov simulation": summary(markov_sim),
    "no memory": summary(iid_sim),
}).round(3)

## Discussion questions

1. Why does checking the return times only up to $k = n$ give the wrong period for the matrix $P$ in exercise 25b?
2. The stationary distribution of the fitted chain is close to the observed wet fraction. Does this show that the rainfall is a Markov chain? Why or why not?
3. Which statistics in the table does the Markov chain reproduce well? Which ones does the model without memory get wrong, and why?
4. Run the simulation cell again: the table changes. How could you decide whether a difference between the observed and the simulated column is real or just randomness?